In [1]:
#import drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Checking whether it can predict the disease or not for a single image...

# Add ML/ to path so we can import src/
import sys
sys.path.append('/content/drive/MyDrive/Multiple Disease Detection Project/ML')

from src.tb_model import TB_CNN
import torch
from torchvision import transforms
from PIL import Image
import os

# Load model
model = TB_CNN()
model.load_state_dict(torch.load('/content/drive/MyDrive/Multiple Disease Detection Project/ML/Models/tb_cnn.pt'))
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Define image transformation (using ImageNet normalization, consistent with evaluation cell 5mPpD6Z_8ohv)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

classes = ['Normal', 'Tuberculosis'] # Ensure these match your model's output classes

def predict_single_tb_image(image_path):
    """Predicts the class of a single TB X-ray image."""
    if not os.path.exists(image_path):
        print(f"Error: Image not found at '{image_path}'. Please check the path.")
        return

    try:
        img = Image.open(image_path).convert('RGB') # Ensure 3 channels
        input_tensor = transform(img).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(input_tensor)
            _, predicted = torch.max(output, 1)
            prediction = classes[predicted.item()]
            print(f"Prediction for {os.path.basename(image_path)}: {prediction}")
            return prediction
    except Exception as e:
        print(f"Could not process image {image_path}: {e}")
        return None

# Example usage for a single image:
# IMPORTANT: Update this path to a specific image you want to predict
single_image_path = '/content/drive/MyDrive/Multiple Disease Detection Project/Datasets/tb merged/tb diagnosed/TB.10.jpg'
print("\n--- Single Image Prediction ---")
predict_single_tb_image(single_image_path)
print("-------------------------------")

# The previous multi-image prediction logic is commented out below.
# If you wish to run multi-image prediction, uncomment the following block
# and update 'prediction_folder' as needed.
#
# prediction_folder = '/content/drive/MyDrive/Dataset of Tuberculosis Chest X-rays Images/TB Chest X-rays' # Example path
#
# if not os.path.exists(prediction_folder):
#     print(f"Error: Prediction folder not found at '{prediction_folder}'. Please update the path.")
# else:
#     image_files = []
#     for f in os.listdir(prediction_folder):
#         f_path = os.path.join(prediction_folder, f)
#         if os.path.isfile(f_path) and f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
#             image_files.append(f_path)
#
#     normal_count = 0
#     tuberculosis_count = 0
#
#     if not image_files:
#         print(f"No image files found in '{prediction_folder}'.")
#     else:
#         print(f"Processing {len(image_files)} images from '{prediction_folder}'...")
#
#         from tqdm.notebook import tqdm # For progress bar
#         for img_path in tqdm(image_files, desc="Predicting images"):
#             try:
#                 img = Image.open(img_path).convert('RGB') # Ensure 3 channels
#                 input_tensor = transform(img).unsqueeze(0).to(device)
#
#                 with torch.no_grad():
#                     output = model(input_tensor)
#                     _, predicted = torch.max(output, 1)
#                     if classes[predicted.item()] == 'Normal':
#                         normal_count += 1
#                     else:
#                         tuberculosis_count += 1
#             except Exception as e:
#                 print(f"Could not process image {img_path}: {e}")
#
#         print("\n--- Prediction Summary ---")
#         print(f"Total images processed: {len(image_files)}")
#         print(f"Normal: {normal_count}")
#         print(f"Tuberculosis: {tuberculosis_count}")
#         print("-------------------------")


--- Single Image Prediction ---
Prediction for TB.10.jpg: Tuberculosis
-------------------------------


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
import os
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image
import seaborn as sns
import matplotlib.pyplot as plt

# Add src folder to sys path
sys.path.append('/content/drive/MyDrive/Multiple Disease Detection Project/ML/src')
from src.oct_model import OCTNet # Your model class

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the model
model = OCTNet().to(device)
model.load_state_dict(torch.load('/content/drive/MyDrive/Multiple Disease Detection Project/ML/Models/oct_cnn.pt'))
model.eval()

# Class names
class_names = ['CNV', 'DME', 'DRUSEN', 'NORMAL']

# -------------------------------
# 🔹 PART 1: Predict a Single Image
# -------------------------------
def predict_single_image(img_path):
    img = Image.open(img_path).convert('RGB')
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])
    input_tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)
        _, predicted = torch.max(output, 1)
        print("🔍 Predicted:", class_names[predicted.item()])


# Example prediction call
img_path = '/content/drive/MyDrive/Multiple Disease Detection Project/Latest_try/bb759b20-1781-437c-bd76-b2816cf2337b.jpg'
predict_single_image(img_path)

# -------------------------------
# 🔹 PART 2: Evaluate Entire Test Dataset
# -------------------------------
def evaluate_test_set():
    test_dir = '/content/drive/MyDrive/Multiple Disease Detection Project/Datasets/Dataset - train+val+test/test'

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])

    test_dataset = datasets.ImageFolder(test_dir, transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Classification Report
    print("\n📊 Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    # Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()


# Example evaluation call
evaluate_test_set()


🔍 Predicted: CNV


FileNotFoundError: Found no valid file for the classes CNV, DME, DRUSEN, NORMAL. Supported extensions are: .jpg, .jpeg, .png, .ppm, .bmp, .pgm, .tif, .tiff, .webp

In [ ]:
!cat "/content/drive/MyDrive/Multiple Disease Detection Project/ML/src/oct_model.py"

import torch.nn as nn
import torch.nn.functional as F

class OCTNet(nn.Module):
    def __init__(self):
        super(OCTNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(32 * 56 * 56, 256)
        self.fc2 = nn.Linear(256, 4)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # -> (16, 112, 112)
        x = self.pool(F.relu(self.conv2(x)))  # -> (32, 56, 56)
        x = x.view(-1, 32 * 56 * 56)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


# Model Evaluation

In [ ]:
# TB model evaluation

import torch
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from PIL import Image
from src.tb_model import TB_CNN
import os

# Define the device to use (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Define the model architecture (or import from src)
model = TB_CNN()
model.to(device)

# 2. Load saved weights
model_path = '/content/drive/MyDrive/Multiple Disease Detection Project/ML/Models/tb_cnn.pt'
model.load_state_dict(torch.load(model_path, map_location=device)) # Added map_location for flexibility
model.eval()

# --- Data Loading and Preparation ---

# Define the path to your validation dataset
val_data_dir = '/content/drive/MyDrive/Multiple Disease Detection Project/Datasets/tb merged'  # <--- **Replace with the actual path**

# Define the transformations for the validation data (should be consistent with training)
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),  # Example: Resize images
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Example: ImageNet normalization
])

# Load the validation dataset using ImageFolder (assuming your data is in subdirectories by class)
try:
    val_dataset = datasets.ImageFolder(root=val_data_dir, transform=val_transforms)
except FileNotFoundError:
    print(f"Error: Validation data directory not found at '{val_data_dir}'. Please update the path.")
    exit()

# Create the DataLoader for the validation dataset
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2) # Adjust batch_size and num_workers as needed

# 3. Evaluate
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Validation Accuracy: {accuracy:.2f}%")

Validation Accuracy: 92.73%


Brain_MRI evaluation


In [ ]:

import torch
from torchvision import transforms, datasets
import sys
import os
sys.path.append(os.path.abspath('/content/drive/MyDrive/Multiple Disease Detection Project/ML'))  # Add current directory to path

from src.brain_mri_model import BrainMRI_CNN  # Now this should work
from torch.utils.data import DataLoader
from PIL import Image
from tqdm.notebook import tqdm  # For progress bar in notebooks

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Cell 2: Model Initialization and Loading Weights

# Instantiate the model
model = BrainMRI_CNN()
model.to(device)

# Path to saved model weights
model_path = '/content/drive/MyDrive/Multiple Disease Detection Project/ML/Models/brain_mri_cnn_v1.pt'

# Load weights
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model weights not found at '{model_path}'. Please check the path.")

model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
print("Model loaded and set to evaluation mode.")

# Cell 3: Data Preparation

# Path to validation data
val_data_dir = '/content/drive/MyDrive/Multiple Disease Detection Project/Datasets/Brain MRI'

# Check if directory exists
if not os.path.isdir(val_data_dir):
    raise FileNotFoundError(f"Validation data directory not found at '{val_data_dir}'. Please update the path.")

# Define transforms (should match training)
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
val_dataset = datasets.ImageFolder(root=val_data_dir, transform=val_transforms)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Validation dataset loaded: {len(val_dataset)} images, {len(val_dataset.classes)} classes.")

# Cell 4: Evaluation

correct = 0
total = 0

# Use tqdm for progress bar in notebook
with torch.no_grad():
    for inputs, labels in tqdm(val_loader, desc="Evaluating", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total if total > 0 else 0
print(f"Validation Accuracy: {accuracy:.2f}%")


Using device: cpu
Model loaded and set to evaluation mode.
Validation dataset loaded: 253 images, 2 classes.


Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

Validation Accuracy: 85.38%


# New Section